# 04 · Directional colocalization analysis

This notebook tests whether annotated cell populations occur together more often than expected and whether those relationships differ between conditions. The configured `analysis_scope` chooses either a genuinely pooled `whole_sample` graph or separate `within_compartment` graphs. Directional results are primary; reciprocal evidence is a stricter secondary confirmation.

Set the scope in `configs/local.yaml`, build that scope, then run the notebook from top to bottom.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from spatial_workflow.colocalization import (
    display_contrast_table,
    display_within_condition_colocalization,
    load_colocalization_outputs,
    plot_directional_contrast_overview,
    plot_reciprocal_contrast_overview,
)
from spatial_workflow.config import load_config, resolve_path
from spatial_workflow.review import (
    available_compartments,
    load_review_outputs,
    plot_reciprocal_celltype_overlap,
)

CONFIG_PATH = REPO_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)
results_root = resolve_path(CONFIG_PATH, config["paths"]["results_root"])
nncomp_output_dir = resolve_path(
    CONFIG_PATH,
    config["nncomp"]["output_dir"],
    root=results_root,
)
colocalization_config = config.get("colocalization", {})
analysis_scope = str(
    colocalization_config.get(
        "analysis_scope", "within_compartment"
    )
)
colocalization_output_root = resolve_path(
    CONFIG_PATH,
    colocalization_config.get(
        "output_dir", "04_colocalization_analysis"
    ),
    root=results_root,
)
colocalization_output_dir = (
    colocalization_output_root / analysis_scope
)
legacy_output = colocalization_output_root / "all_directional_limma.csv"
if (
    analysis_scope == "within_compartment"
    and not colocalization_output_dir.exists()
    and legacy_output.exists()
):
    colocalization_output_dir = colocalization_output_root

nncomp_tables = load_review_outputs(nncomp_output_dir)
colocalization_tables = load_colocalization_outputs(
    colocalization_output_dir
)
permutation_key = f"{analysis_scope}_celltype_permutation"
celltype_permutation = nncomp_tables.get(permutation_key)
if celltype_permutation is None:
    raise FileNotFoundError(
        f"Missing {permutation_key}. Run the all-cell-type nncomp stage first."
    )
contrast_results = colocalization_tables["contrast_results"]

## Review parameters

Edit the controls below and rerun the downstream display cells. These filters change only the displayed review; they do not modify the complete result tables.

Rebuild after changing the scope, neighborhood settings, eligibility rules, or planned contrasts. Each scope is written to its own output subdirectory:

```bash
python3 scripts/build_colocalization_tables.py \
  --config configs/local.yaml --analysis-scope whole_sample --overwrite
```

In [ ]:
review_config = config.get("review", {})
colocalization_review = colocalization_config.get("review", {})
domains = available_compartments(nncomp_tables["compartment_abundance"])

# Shared scope and support parameters.
selected_domain = (
    "all"
    if analysis_scope == "whole_sample"
    else str(review_config.get("selected_domain") or domains[0])
)
contrast_compartments = (
    ["all"] if analysis_scope == "whole_sample" else None
)
selected_k = int(
    colocalization_config.get("k", review_config.get("overlap_k", 2))
)
min_cells_a = int(
    colocalization_config.get("min_cells_a", review_config.get("min_overlap_cells_a", 10))
)
min_cells_b = int(
    colocalization_config.get("min_cells_b", review_config.get("min_overlap_cells_b", 10))
)
expected_samples = int(colocalization_config.get("expected_samples", 4))

# Within-condition evidence filters.
within_min_positive_samples = 3
within_min_median_delta = 0.0
within_max_stouffer_q = 0.15
within_max_partial_q = 0.10

# Require at least 5% observed directional overlap in one contrast condition.
min_observed_colocalization = float(
    colocalization_review.get("min_observed_colocalization", 0.05)
)
contrast_plot_top_n = 15
contrast_plot_max_exact_p = 0.05
contrast_plot_min_loo_sign_fraction = 1.0
require_within_condition_colocalization = True
show_reciprocal_confirmation = True
show_z_diagnostics = False
selected_contrast = (
    "vap_igg_vs_naive_igg"
    if "vap_igg_vs_naive_igg" in contrast_results
    else next(iter(contrast_results))
)

# Reciprocal Plotly filters.
plot_cell_types_a = review_config.get("overlap_cell_types_a")
plot_cell_types_b = review_config.get("overlap_cell_types_b")
plot_min_samples = int(review_config.get("min_overlap_samples", expected_samples))
plot_min_abs_z_score = float(review_config.get("min_abs_z_score", 0.0))
plot_z_direction = str(review_config.get("z_direction", "both"))

review_parameters = {
    "analysis scope": analysis_scope,
    "selected spatial context": selected_domain,
    "k": selected_k,
    "minimum cells A": min_cells_a,
    "minimum cells B": min_cells_b,
    "minimum observed colocalization": min_observed_colocalization,
    "maximum exact permutation p": contrast_plot_max_exact_p,
    "require positive within-condition colocalization": require_within_condition_colocalization,
    "selected contrast": selected_contrast,
}
display(pd.Series(review_parameters, name="review parameters").to_frame())

## 1. Directional colocalization within each condition

For each source-to-target relationship, the table reports observed overlap and enrichment above the null for the selected analysis scope. `within_compartment` uses separate domain graphs and shuffles labels within sample-domain strata. `whole_sample` pools all cells in each biological sample and shuffles labels within that sample without using compartments. Direction is retained because the source population defines the denominator.

Use this section to establish positive colocalization before interpreting a between-condition difference.

In [ ]:
example_directional_within_condition = (
    display_within_condition_colocalization(
        colocalization_tables,
        table="directional",
        compartments=[selected_domain],
        k_values=[selected_k],
        min_samples=expected_samples,
        min_positive_samples=within_min_positive_samples,
        include_self_pairs=False,
        min_condition_cells=max(min_cells_a, min_cells_b),
        min_median_delta=within_min_median_delta,
        max_stouffer_q=within_max_stouffer_q,
        max_partial_q=within_max_partial_q,
        n=50,
    )
)

## 2. Directional changes between conditions

The primary plot compares condition-specific overlap for relationships that are positively colocalized in at least one contrast condition. This prevents a significant difference between two depleted relationships from being labeled as colocalization.

**Score guide:**

- **Mean coefficient** is the condition mean of the fraction of source cells having at least one target cell among their first `k` neighbors. `coefficient_change` is the raw numerator-minus-denominator change in that fraction.
- **Mean delta** is the condition mean of observed coefficient minus its within-sample, within-domain label-permutation expectation. `contrast_delta` is numerator mean delta minus denominator mean delta; positive values favor the numerator condition.
- `exact_permutation_p` is the two-sided, exhaustive test obtained by reassigning the biological samples to the two conditions and recomputing `contrast_delta`. It permutes sample labels, not cells, and is an unadjusted p-value. With four samples per condition its smallest attainable two-sided value is `0.02857`.
- `limma_p` is the empirical-Bayes sample-level test of the same differential delta, and `limma_q` is its multiple-testing-adjusted value.
- `loo_sign_fraction` is the fraction of leave-one-sample-out comparisons with the same effect direction as the full comparison; `1.0` means the sign survives every omission.
- The condition colocalization flags indicate whether the complete positive-colocalization evidence gate passed in that condition; a coefficient or small between-condition p-value alone is not sufficient.

Use effect size, exact p-value, adjusted evidence, and leave-one-out stability together when selecting relationships for follow-up.

In [ ]:
directional_contrast_figures = {}
for contrast in contrast_results:
    figure = plot_directional_contrast_overview(
        colocalization_tables,
        contrast,
        top_n=contrast_plot_top_n,
        rank_by="max_abs_contrast",
        compartments=contrast_compartments,
        min_observed_colocalization=min_observed_colocalization,
        require_within_condition_colocalization=(
            require_within_condition_colocalization
        ),
        within_min_positive_samples=within_min_positive_samples,
        within_min_median_delta=within_min_median_delta,
        within_max_stouffer_q=within_max_stouffer_q,
        within_max_partial_q=within_max_partial_q,
        max_exact_p=contrast_plot_max_exact_p,
        min_loo_sign_fraction=(
            contrast_plot_min_loo_sign_fraction
        ),
    )
    directional_contrast_figures[contrast] = figure
    display(
        pd.Series(
            figure.layout.meta,
            name=f"{contrast}: directional",
        ).to_frame()
    )
    figure.show()

### Directional result table

The table places the two condition coefficients first, followed by colocalization flags, the between-condition effect, exact permutation statistics, and stability measures.

In [ ]:
example_filtered = display_contrast_table(
    colocalization_tables,
    selected_contrast,
    table="directional",
    compartments=contrast_compartments,
    cell_types_a=None,
    min_observed_colocalization=min_observed_colocalization,
    require_within_condition_colocalization=(
        require_within_condition_colocalization
    ),
    within_min_positive_samples=within_min_positive_samples,
    within_min_median_delta=within_min_median_delta,
    within_max_stouffer_q=within_max_stouffer_q,
    within_max_partial_q=within_max_partial_q,
    max_exact_p=contrast_plot_max_exact_p,
    min_loo_sign_fraction=contrast_plot_min_loo_sign_fraction,
    include_sample_counts=False,
    n=50,
)

## 3. Reciprocal confirmation

Reciprocal evidence requires positive colocalization in both directions. It is shown as a secondary confirmation so directional relationships are not discarded simply because the reverse direction is weaker.

In [ ]:
reciprocal_contrast_figures = {}
if show_reciprocal_confirmation:
    for contrast in contrast_results:
        figure = plot_reciprocal_contrast_overview(
            colocalization_tables,
            contrast,
            top_n=contrast_plot_top_n,
            rank_by="max_abs_contrast",
            compartments=contrast_compartments,
            reciprocal_same_direction=True,
            min_observed_colocalization=(
                min_observed_colocalization
            ),
            require_within_condition_colocalization=(
                require_within_condition_colocalization
            ),
            within_min_positive_samples=(
                within_min_positive_samples
            ),
            within_min_median_delta=within_min_median_delta,
            within_max_stouffer_q=within_max_stouffer_q,
            within_max_partial_q=within_max_partial_q,
            max_exact_p=contrast_plot_max_exact_p,
            min_loo_sign_fraction=(
                contrast_plot_min_loo_sign_fraction
            ),
        )
        reciprocal_contrast_figures[contrast] = figure
        display(
            pd.Series(
                figure.layout.meta,
                name=f"{contrast}: reciprocal",
            ).to_frame()
        )
        figure.show()
else:
    print("Set show_reciprocal_confirmation=True to render this view.")

In [ ]:
reciprocal_detail = display_contrast_table(
    colocalization_tables,
    selected_contrast,
    table="reciprocal",
    compartments=contrast_compartments,
    reciprocal_same_direction=True,
    min_observed_colocalization=min_observed_colocalization,
    require_within_condition_colocalization=(
        require_within_condition_colocalization
    ),
    within_min_positive_samples=within_min_positive_samples,
    within_min_median_delta=within_min_median_delta,
    within_max_stouffer_q=within_max_stouffer_q,
    within_max_partial_q=within_max_partial_q,
    max_exact_p=contrast_plot_max_exact_p,
    min_loo_sign_fraction=contrast_plot_min_loo_sign_fraction,
    n=20,
)

## 4. Optional reciprocal z-score diagnostic

This diagnostic compares the two directional permutation z-scores for each pair. Positive values indicate more neighboring cells than expected under the selected scope-specific null; negative values indicate spatial depletion.

Enable this section when the geometry of both directions is useful for interpreting a reciprocal result.

### Sample-level reciprocal view

Points show the two directional z-scores for individual samples and condition means. Hover labels provide the analysis scope, spatial context, pair, support counts, and both directional statistics.

In [ ]:
if not show_z_diagnostics:
    print("Set show_z_diagnostics=True to render the z-score view.")

In [ ]:
if show_z_diagnostics:
    selected_compartment_figure = plot_reciprocal_celltype_overlap(
        celltype_permutation,
        compartment=selected_domain,
        analysis_scope=analysis_scope,
        cell_types_a=plot_cell_types_a,
        cell_types_b=plot_cell_types_b,
        k_value=selected_k,
        min_cells_a=min_cells_a,
        min_cells_b=min_cells_b,
        min_samples=plot_min_samples,
        min_abs_z_score=plot_min_abs_z_score,
        z_direction=plot_z_direction,
    )
    display(
        pd.Series(
            selected_compartment_figure.layout.meta,
            name=(
                "whole sample"
                if analysis_scope == "whole_sample"
                else f"compartment {selected_domain}"
            ),
        ).to_frame()
    )
    selected_compartment_figure.show()